# Confidence Threshold Analysis

This notebook analyzes confidence scores to find the optimal threshold for fallback triggering.

In [ ]:
# Cell 1: Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
MODEL_DIR = PROJECT_ROOT / "models" / "intent_classifier_v2"

print(f"Loading results from: {MODEL_DIR}")

In [ ]:
# Cell 2: Load test results
results_df = pd.read_csv(MODEL_DIR / 'test_results.csv')
print(f"Loaded {len(results_df)} test results")
print(f"\nColumns: {results_df.columns.tolist()}")

In [ ]:
# Cell 3: Confidence statistics
correct = results_df[results_df['correct'] == True]
wrong = results_df[results_df['correct'] == False]

print("Confidence Statistics:")
print("=" * 50)
print(f"\nCorrect predictions ({len(correct)}):")
print(f"  Mean: {correct['confidence'].mean():.4f}")
print(f"  Std:  {correct['confidence'].std():.4f}")
print(f"  Min:  {correct['confidence'].min():.4f}")
print(f"  Max:  {correct['confidence'].max():.4f}")

if len(wrong) > 0:
    print(f"\nWrong predictions ({len(wrong)}):")
    print(f"  Mean: {wrong['confidence'].mean():.4f}")
    print(f"  Std:  {wrong['confidence'].std():.4f}")
    print(f"  Min:  {wrong['confidence'].min():.4f}")
    print(f"  Max:  {wrong['confidence'].max():.4f}")

In [ ]:
# Cell 4: Threshold analysis
print("\nThreshold Analysis:")
print("=" * 70)
print(f"{'Threshold':<12} {'Accuracy':<12} {'Coverage':<12} {'Fallback %':<12}")
print("-" * 70)

thresholds = np.arange(0.50, 0.96, 0.05)
threshold_results = []

for thresh in thresholds:
    above_thresh = results_df[results_df['confidence'] >= thresh]
    below_thresh = results_df[results_df['confidence'] < thresh]
    
    if len(above_thresh) > 0:
        acc_above = above_thresh['correct'].mean()
        coverage = len(above_thresh) / len(results_df)
        fallback_rate = len(below_thresh) / len(results_df)
        
        threshold_results.append({
            'threshold': thresh,
            'accuracy': acc_above,
            'coverage': coverage,
            'fallback_rate': fallback_rate
        })
        
        print(f"{thresh:<12.2f} {acc_above:<12.4f} {coverage:<12.4f} {fallback_rate:<12.4f}")

threshold_df = pd.DataFrame(threshold_results)

In [ ]:
# Cell 5: Find optimal threshold
# Goal: Accuracy > 90% with coverage > 80%

print("\nFinding Optimal Threshold:")
print("=" * 50)

# Filter for high accuracy
high_acc = threshold_df[threshold_df['accuracy'] >= 0.90]

if len(high_acc) > 0:
    # Among high accuracy, get highest coverage
    optimal = high_acc.loc[high_acc['coverage'].idxmax()]
    print(f"\nOptimal threshold (acc >= 90%):")
    print(f"  Threshold: {optimal['threshold']:.2f}")
    print(f"  Accuracy: {optimal['accuracy']:.4f}")
    print(f"  Coverage: {optimal['coverage']:.4f}")
    print(f"  Fallback rate: {optimal['fallback_rate']:.4f}")
    
    RECOMMENDED_THRESHOLD = optimal['threshold']
else:
    # Use threshold with best F1-like balance
    threshold_df['f1_like'] = 2 * (threshold_df['accuracy'] * threshold_df['coverage']) / (threshold_df['accuracy'] + threshold_df['coverage'])
    optimal = threshold_df.loc[threshold_df['f1_like'].idxmax()]
    print(f"\nOptimal threshold (balanced):")
    print(f"  Threshold: {optimal['threshold']:.2f}")
    print(f"  Accuracy: {optimal['accuracy']:.4f}")
    print(f"  Coverage: {optimal['coverage']:.4f}")
    
    RECOMMENDED_THRESHOLD = optimal['threshold']

print(f"\nRECOMMENDED THRESHOLD: {RECOMMENDED_THRESHOLD:.2f}")

In [ ]:
# Cell 6: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy vs Coverage trade-off
ax1 = axes[0]
ax1.plot(threshold_df['threshold'], threshold_df['accuracy'], 'b-o', label='Accuracy', linewidth=2)
ax1.plot(threshold_df['threshold'], threshold_df['coverage'], 'g-s', label='Coverage', linewidth=2)
ax1.axvline(x=RECOMMENDED_THRESHOLD, color='red', linestyle='--', label=f'Recommended ({RECOMMENDED_THRESHOLD:.2f})')
ax1.axhline(y=0.90, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Score')
ax1.set_title('Accuracy vs Coverage Trade-off')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Confidence distribution
ax2 = axes[1]
ax2.hist(correct['confidence'], bins=20, alpha=0.7, label='Correct', color='green')
if len(wrong) > 0:
    ax2.hist(wrong['confidence'], bins=20, alpha=0.7, label='Wrong', color='red')
ax2.axvline(x=RECOMMENDED_THRESHOLD, color='blue', linestyle='--', label=f'Threshold ({RECOMMENDED_THRESHOLD:.2f})')
ax2.set_xlabel('Confidence Score')
ax2.set_ylabel('Count')
ax2.set_title('Confidence Distribution')
ax2.legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'threshold_analysis.png', dpi=300)
plt.show()

print(f"\nFigure saved to: {MODEL_DIR / 'threshold_analysis.png'}")

In [ ]:
# Cell 7: Update model config with recommended threshold
label_mapping_path = MODEL_DIR / "best_model" / "label_mapping.json"

with open(label_mapping_path, 'r') as f:
    label_mapping = json.load(f)

# Update threshold
label_mapping['confidence_threshold'] = float(RECOMMENDED_THRESHOLD)

with open(label_mapping_path, 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"Updated label_mapping.json with threshold: {RECOMMENDED_THRESHOLD:.2f}")

# Save threshold analysis results
threshold_df.to_csv(MODEL_DIR / 'threshold_analysis.csv', index=False)
print(f"Threshold analysis saved to: {MODEL_DIR / 'threshold_analysis.csv'}")

In [ ]:
# Cell 8: Summary
print("\n" + "=" * 60)
print("CONFIDENCE THRESHOLD ANALYSIS COMPLETE")
print("=" * 60)

print(f"""
Summary:
--------
Recommended Threshold: {RECOMMENDED_THRESHOLD:.2f}

At this threshold:
- Accuracy: {optimal['accuracy']:.2%}
- Coverage: {optimal['coverage']:.2%}
- Fallback rate: {optimal['fallback_rate']:.2%}

How it works:
- If confidence >= {RECOMMENDED_THRESHOLD:.2f}: Return prediction
- If confidence < {RECOMMENDED_THRESHOLD:.2f}: Trigger fallback (WhatsApp redirect)

Files updated:
- {label_mapping_path}
- {MODEL_DIR / 'threshold_analysis.csv'}
- {MODEL_DIR / 'threshold_analysis.png'}
""")